# Dynamic Pricing Env with Lag 

> Overarching DP env

In [ ]:
#| default_exp envs.pricing.dynamic_RL2

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from abc import ABC, abstractmethod
from typing import Union, Tuple, Literal

from ddopai.utils import Parameter, MDPInfo
from ddopai.dataloaders.base import BaseDataLoader
from ddopai.loss_functions import pinball_loss, quantile_loss
from ddopai.envs.pricing.base import BasePricingEnv
import gymnasium as gym

import numpy as np
import time

In [ ]:
#| export

class RL2DynamicPricingEnv(BasePricingEnv):
    """
    Dynamic Pricing Environment adapted for RL².
    Observation = (current features, current inventory, previous action, reward, done).
    """

    def __init__(self,
                alpha: Union[np.ndarray, Parameter, int, float] = 1.0,  # market size
                beta: Union[np.ndarray, Parameter, int, float] = 0.5,   # price elasticity
                p_bound_low: Union[np.ndarray, Parameter, int, float] = 0.0,  # lower price bound
                p_bound_high: Union[np.ndarray, Parameter, int, float] = 1.0, # upper price bound
                dataloader: BaseDataLoader = None,  # dataloader TODO: replace with pricing dataloader
                gamma: float = 1,  # discount factor
                
                nb_features: int = 1,  # number of features
                
                covariance: Union[np.ndarray, Parameter, int, float] = 1,  # standard deviation of the features
                noise_std: Union[np.ndarray, Parameter, int, float] = 1,     # standard deviation of the noise
                function_form: Union[np.ndarray, Parameter, str] = "linear", # functional form of the demand function
                inv: Union[np.ndarray, Parameter, int, float] = 100,         # inventory
                horizon_train: int | str = "use_all_data",  # if "use_all_data", then horizon is inferred from the DataLoader
                postprocessors: list[object] | None = None,  # default is empty list 
                mode: str = "train", 
                return_truncation: str = False,  # TODO:Why is this a string?
                env_type: dict = {"inv": False, "reference_price": False},
                ) -> None:
        self.print = False

        if dataloader is None:
            dataloader = self.update_dataloader()
            
        # Ignore the multi-SKU feature: always use one SKU.
        num_SKUs = 1  
        # Remove SKU from parameters.
        # self.set_param("num_SKUs", num_SKUs, new=True)   <-- removed
        
        self.set_param("alpha", alpha, shape=np.atleast_1d(alpha).shape, new=True)
        self.set_param("beta", beta, shape=np.atleast_1d(beta).shape, new=True)
        # Force p_bound_low and p_bound_high to be 1D arrays of length 1.
        self.set_param("p_bound_low", p_bound_low, shape=(1,), new=True)
        self.set_param("p_bound_high", p_bound_high, shape=(1,), new=True)
        
        self.set_param("nb_features", nb_features, new=True)
        if isinstance(covariance, np.ndarray):
            self.set_param("covariance", covariance, shape=covariance.shape, new=True)
        else:
            self.set_param("covariance", covariance, new=True)
        if isinstance(noise_std, np.ndarray):
            self.set_param("noise_std", noise_std, shape=noise_std.shape, new=True)
        else:
            self.set_param("noise_std", noise_std, new=True)
        self.set_param("function_form", function_form, new=True)
        
        # Inventory parameters: use the first element.
        self.set_param("inv", inv[0], inv[0].shape, new=True)
        relative_inv = inv[0].copy()
        relative_inv[-1] = 1.0  # np.float64(1.0)
        self.set_param("relative_inv", relative_inv, relative_inv.shape, new=True)
        self.set_param("inv_per_episode", inv, inv.shape, new=True)
        self.set_param("horizon_train", horizon_train, new=True)

        self._prev_action = np.zeros((1,), dtype=np.float32)
        self._prev_reward = np.zeros((1,), dtype=np.float32)
        self._prev_done = np.ones((1,), dtype=np.float32)

        low = np.min(dataloader.X, axis=0)
        high = np.max(dataloader.X, axis=0)
        feature_shape = dataloader.X_shape[1:]
        # Set the observation space without the SKU dimension.
        self.set_observation_space(feature_shape=feature_shape, feature_low=low, feature_high=high)
        self.set_action_space(dataloader.Y_shape, low=self.p_bound_low, high=self.p_bound_high)
        
        mdp_info = MDPInfo(self.observation_space, self.action_space, gamma=gamma, horizon=horizon_train)
        
        super().__init__(mdp_info=mdp_info,
                         postprocessors=postprocessors,
                         mode=mode, return_truncation=return_truncation,
                         dataloader=dataloader,
                         horizon_train=horizon_train)

    def set_observation_space(self, feature_shape, feature_low, feature_high):
        """
        Define the observation space.
        """
        spaces = {
            "features": gym.spaces.Box(
                low=feature_low,
                high=feature_high,
                shape=feature_shape,
                dtype=np.float32
            ),
            "inventory": gym.spaces.Box(
                low=0.0,
                high=1.0,
                shape=(1,),
                dtype=np.float32
            ),
            "prev_action": gym.spaces.Box(
                low=self.p_bound_low,
                high=self.p_bound_high,
                shape=(1,),
                dtype=np.float32
            ),
            "prev_reward": gym.spaces.Box(
                low=-np.inf,
                high=np.inf,
                shape=(1,),
                dtype=np.float32
            ),
            "prev_done": gym.spaces.Box(
                low=0.0,
                high=1.0,
                shape=(1,),
                dtype=np.float32
            )
        }
        self.observation_space = gym.spaces.Dict(spaces)

    def step_(self, action: np.ndarray):
        """
        Step forward in the environment.
        """
        if action.ndim == 2 and action.shape[0] == 1:
            action = np.squeeze(action, axis=0)

        observation, reward_functions = self.get_observation()

        x = observation["features"]
        demand, demand_noise_free = reward_functions[0](x, action)

        if self.env_type["inv"]:
            if (demand / self.inv) >= self.relative_inv:
                demand = self.relative_inv * self.inv
                self.relative_inv = 0
            else:
                self.relative_inv -= demand / self.inv
        
        reward = demand * action

        terminated = self.relative_inv == 0
        truncated = self.set_index()

        info = dict(
            inv=self.inv * self.relative_inv,
            demand=demand,
            demand_noise_free=demand_noise_free,
            action=action.copy(),
            reward=reward
        )

        # Save previous
        self._prev_action = np.array([action], dtype=np.float32)
        self._prev_reward = np.array([reward], dtype=np.float32)
        self._prev_done = np.array([float(terminated or truncated)], dtype=np.float32)

        if truncated:
            if self.mode in ["test", "val"]:
                observation = None
            else:
                observation, _ = self.get_observation()
            return observation, reward, terminated, truncated, info
        else:
            observation, _ = self.get_observation()
            if self.print:
                print("next_period:", self.index + 1)
                print("next observation:", observation)
                time.sleep(3)
            return observation, reward, terminated, truncated, info

    def get_observation(self):
        """
        Build the observation dict.
        """
        x, reward_functions = self.dataloader[self.index]
        current_inv = np.array([self.relative_inv], dtype=np.float32)

        observation = {
            "features": x,
            "inventory": current_inv,
            "prev_action": self._prev_action,
            "prev_reward": self._prev_reward,
            "prev_done": self._prev_done
        }
        return observation, reward_functions

    def reset(self, start_index=None, state=None):
        """
        Reset environment to initial state.
        """
        truncated = self.reset_index(start_index)

        self.relative_inv = self.inv

        self._prev_action = np.zeros((1,), dtype=np.float32)
        self._prev_reward = np.zeros((1,), dtype=np.float32)
        self._prev_done = np.ones((1,), dtype=np.float32)

        observation, _ = self.get_observation()
        return observation

    def reset_env(self, epoch):
        pass
